The purpose of this file is to analyze and detect an up coming recession based on the data pulled from https://fred.stlouisfed.org/

In [49]:
import matplotlib.pyplot as plt
import pandas as pd
from fredapi import Fred
import datetime
import numpy as np
import random
import plotly.express as px

FRED_API_KEY = "612c90fb30af12767bbcaf9513bac5ed"
fred = Fred(api_key=FRED_API_KEY)

In [50]:
"""Tests the plotting of plotly express"""

df = px.data.stocks()
print(df)

fig = px.line(
    df,
    x="date",
    y=df.columns,
    hover_data={"date": "|%B %d, %Y"},
    title="custom tick labels",
    template="plotly_dark",
    width=1000,
    height=400,
)

fig.update_xaxes(dtick="M1", tickformat="%b\n%Y", rangeslider_visible=True)
fig.show()

           date      GOOG      AAPL      AMZN        FB      NFLX      MSFT
0    2018-01-01  1.000000  1.000000  1.000000  1.000000  1.000000  1.000000
1    2018-01-08  1.018172  1.011943  1.061881  0.959968  1.053526  1.015988
2    2018-01-15  1.032008  1.019771  1.053240  0.970243  1.049860  1.020524
3    2018-01-22  1.066783  0.980057  1.140676  1.016858  1.307681  1.066561
4    2018-01-29  1.008773  0.917143  1.163374  1.018357  1.273537  1.040708
..          ...       ...       ...       ...       ...       ...       ...
100  2019-12-02  1.216280  1.546914  1.425061  1.075997  1.463641  1.720717
101  2019-12-09  1.222821  1.572286  1.432660  1.038855  1.421496  1.752239
102  2019-12-16  1.224418  1.596800  1.453455  1.104094  1.604362  1.784896
103  2019-12-23  1.226504  1.656000  1.521226  1.113728  1.567170  1.802472
104  2019-12-30  1.213014  1.678000  1.503360  1.098475  1.540883  1.788185

[105 rows x 7 columns]


In [51]:
def get_recession_data() -> pd.DataFrame:
    """
    Creates the recession dates dataframe

    Returns:
    - pd.DataFrame: The recession dates dataframe.
    """
    df_recession = pd.read_csv("data/USREC.csv")
    df_recession = df_recession.rename(columns={"USREC": "recession", "DATE": "date"})
    df_recession["date"] = pd.to_datetime(df_recession["date"])
    df_recession = df_recession.set_index("date")
    return df_recession


recdf = get_recession_data()
recdf.reset_index(inplace=True)

fig = px.line(
    recdf,
    x="date",
    y="recession",
    hover_data={"date": "|%B %d, %Y"},
    title="custom tick labels",
    template="plotly_dark",
    width=1000,
    height=400,
)
fig.show()

In [52]:
def merge_recession_data(df: pd.DataFrame) -> pd.DataFrame:
    """
    Creates the recession dates dataframe and
    modifies the start and end dates to match the dataframe passed in.

    Parameters:
    - df (pd.DataFrame): The input dataframe containing the data.

    Returns:
    - pd.DataFrame: The modified recession dates dataframe.
    """

    df_recession = get_recession_data()

    # Make sure that both dataframes start and end at the same index
    start_date = max(df.index.min(), df_recession.index.min())
    end_date = min(df.index.max(), df_recession.index.max())

    # df = df.loc[start_date:end_date]
    df_recession = df_recession.loc[start_date:end_date]

    return df_recession

In [53]:
def get_data_after_date(
    data: pd.DataFrame, years: int = 1, days: int = 0
) -> pd.DataFrame:
    data_clean = data.dropna()

    # get the most recent date, convert to datetime obj and then get 1 year ago date
    most_recent_dt = data_clean.index[-1]
    one_year_ago_dt = most_recent_dt - pd.DateOffset(years=years, days=days)
    return data_clean.loc[one_year_ago_dt:]

In [54]:
def get_recession_start_end_list(recdf):
    reclist = []
    recdf.reset_index(inplace=True)

    # loop through the recession dataframe and find the start and end dates for each recession
    startdate, enddate = None, None
    for i in range(0, len(recdf)):
        if recdf.iloc[i, 1] == 1:  # recession detected
            startdate = recdf.iloc[i, 0]

            for j in range(i, len(recdf)):
                if recdf.iloc[j, 1] == 0:  # end of the recession
                    enddate = recdf.iloc[j, 0]
                    break

        if startdate and enddate:
            if (
                reclist and enddate == reclist[-1][1]
            ):  # if the enddate is the same as the last one, skip
                continue

            reclist.append((startdate, enddate))

            # after adding startdate and enddate to the list,
            # jump to the enddate and start iterating from there
            # i = j

            startdate, enddate = None, None
    return reclist


def plot_it(data_df, title, plot_recession_dates=False):
    data_df.reset_index(inplace=True)

    fig = px.line(
        data_df,
        x="date",
        y=data_df.columns,
        hover_data={"date": "|%B %d, %Y"},
        title=title,
        template="plotly_dark",
        width=1000,
        height=400,
    )

    if plot_recession_dates:
        recdf = get_recession_data()

        start_date = data_df.date.min()
        end_date = data_df.date.max()

        # ensure that the recession dataframe only has dates that are in the data dataframe
        recdf = recdf.loc[(recdf.index >= start_date) & (recdf.index <= end_date)]

        for row in get_recession_start_end_list(recdf):
            x0 = str(row[0].date())
            x1 = str(row[1].date())

            fig.add_vrect(
                x0=x0,
                x1=x1,
                fillcolor="red",
                opacity=0.25,
                line_width=0,
                # annotation_text="recession",
                # annotation_position="top left",
            )
    fig.show()


# data_df = px.data.stocks()
# plot_it(data_df, title="Stocks", plot_recession_dates=True)

In [55]:
def plot_data(symbol, title, years, plot_recession_dates=True):
    data = fred.get_series(symbol)
    result = get_data_after_date(data, years=years)

    if isinstance(result, pd.Series):
        result = pd.DataFrame(result).reset_index()
        result.columns = ["date", "data"]
        result = result.set_index("date")

    plot_it(result, title, plot_recession_dates=plot_recession_dates)

In [56]:
def test_recession_dates(use_random_timeframe: bool = False) -> None:
    """
    Ensures that data can be plotted alongside of the recession dates in one figure
    """

    end = datetime.datetime.now()
    date = datetime.datetime(1985, 1, 1)
    rec_df = get_recession_data()

    if use_random_timeframe:
        # pick a random date in the available data
        # get the first date in the recession dates
        res = random.choices(rec_df.index, k=2)
        rand_years = [i.year for i in res]
        start_year, end_year = min(rand_years), max(rand_years)
        date = datetime.datetime(start_year, 1, 1)
        end = datetime.datetime(end_year, 1, 1)

    date_list = [date]

    while date < end:
        date += datetime.timedelta(days=1)
        date_list.append(date)

    rand_data = np.random.uniform(low=0, high=1, size=len(date_list))
    data_df = pd.DataFrame({"date": date_list, "data": rand_data})

    # Tests if the recession dates were plotted correctly
    data_df = data_df.set_index("date")

    # combine the df with the recession dates
    plot_it(data_df, "Data & Recession Dates", plot_recession_dates=True)


test_recession_dates(use_random_timeframe=True)

In [57]:
# """Federal Funds Effective Rate (FEDFUNDS)"""
plot_data("FEDFUNDS", "Federal Funds Effective Rate", years=70)

In [58]:
"""
GDP Contraction: One of the primary indicators is a decline in Gross Domestic Product (GDP),
which measures the total value of goods and services produced in a country.
A negative GDP growth for two consecutive quarters is often considered a technical recession.
"""

# Real Gross Domestic Product (GDPC1)
plot_data("GDPC1", "Real GDP", years=2)

In [59]:
"""
Rising Unemployment: Job losses and rising unemployment rates are common
during a recession as businesses may cut costs by reducing their workforce.
"""

plot_data("UNRATE", "Unemployment Rate", years=2)

In [60]:
"""
Consumer spending tends to decrease during economic
downturns as people become more cautious about their finances.
This can impact various industries, especially retail.
"""

plot_data("PCE", "Consumer Spending", years=4)

In [61]:
"""
Reduced Industrial Production: A decrease in the production of goods and services
by industries is a clear sign of economic contraction.
This can be measured by the Industrial Production Index.
"""

plot_data("INDPRO", "Industrial Production", years=5)

In [62]:
"""
Stock Market Decline: Stock markets are sensitive to economic conditions.
A prolonged period of declining stock prices may indicate investor pessimism about the economic outlook.
"""

plot_data("SP500", "S&P 500", years=2)

In [63]:
"""
Housing Market Slowdown: A recession often leads to a slowdown in the real estate market,
with declining home sales, falling prices, and increased foreclosures.
"""

plot_data("CSUSHPINSA", "Housing Market", years=4)

In [64]:
"""
Tightened Credit Conditions: Banks may become more cautious about lending during economic downturns,
leading to tightened credit conditions and reduced access to loans for businesses and consumers.
"""

plot_data("DRTSCILM", "Tightened Credit Conditions", years=50)

In [65]:
"""
Inverted Yield Curve: An inverted yield curve, where short-term interest rates are higher
than long-term rates has historically been a reliable predictor of economic recessions.
"""

plot_data("T10Y2Y", "Yield Curve 10Y - 2Y", years=2)

In [66]:
"""10-Year Treasury Constant Maturity Minus 3-Month Treasury Constant Maturity (T10Y3M)"""

plot_data("T10Y3M", "Yield Curve 10Y - 3M", years=1)

In [67]:
"""
Business Investment Decline: Companies may cut back on investments during a recession,
leading to a decline in capital expenditures and business expansion.
"""

plot_data("W790RC1Q027SBEA", "Business Investment", years=1)

In [68]:
plot_data("CEU4348400001", "Truck Transportation Employees", years=1)

In [69]:
"""Delinquency Rate on Credit Card Loans, All Commercial Banks"""

plot_data("DRCCLACBS", "Credit Card Delinquency Rate", years=2)

In [70]:
plot_data("JTSJOL", "Job Openings", years=6)

In [71]:
"""Large Bank Consumer Credit Card Balances: Total Balances (RCCCBBALTOT)"""

plot_data(
    "RCCCBBALTOT", "Large Bank Consumer Credit Card Balances: Total Balances", years=1
)

In [72]:
"""Commercial Bank Interest Rate on Credit Card Plans, All Accounts (TERMCBCCALLNS)"""

plot_data(
    "TERMCBCCALLNS", "Commercial Bank Interest Rate on Credit Card Plans", years=2
)

In [73]:
#  Personal interest payments (B069RC1)
plot_data("B069RC1", "Personal interest payments", years=5)

Other Indicators to Consider
- Net interest and miscellaneous payments on assets
- Personal Savings Rate

In [74]:
#  Net interest and miscellaneous payments on assets (W255RC1Q027SBEA)
plot_data(
    "W255RC1Q027SBEA", "Net interest and miscellaneous payments on assets", years=10
)

In [75]:
#  Personal Saving Rate (PSAVERT)
plot_data("PSAVERT", "Personal Saving Rate", years=6)

In [76]:
# LOOK AT THESE STOCKS
# In addition, certain individual stocks have outperformed during each of the past two U.S. recessions.
# Walmart Inc. (ticker: WMT), Abbott Laboratories (ABT) and Home Depot Inc. (HD)
# are just three examples of stocks that beat the S&P 500 in both 2008 and 2020.